# SPEC2RTL & PROMPT2SILICON: HUẤN LUYỆN SFT LORA TỪ DỮ LIỆU CHƯNG CẤT GEMINI TEACHER
### Đề tài: Hệ thống sinh và kiểm chứng mã Verilog từ đặc tả ngôn ngữ tự nhiên
**Nhóm 2:** Huấn luyện AI Cục bộ (Fine-tuning SFT LoRA trên dữ liệu đã qua ModelSim Truth Oracle)
**Mục tiêu:** Chọn kích thước Student phù hợp phần cứng (`0.5B` khuyên dùng cho máy 4GB RAM; `1.5B` hoặc `3B` cho máy mạnh), nạp **1,191 mẫu sạch** (gồm CoT thật chưng cất từ Google Gemini + ModelSim Oracle) và huấn luyện SFT LoRA trên T4 GPU miễn phí.

## Bước 1: Kiểm tra cấu hình GPU Google Colab
*Lưu ý: Vào menu Runtime -> Change runtime type -> Chọn T4 GPU (miễn phí).*

In [ ]:
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Bước 2: Cài đặt thư viện Unsloth (Tăng tốc độ train x2 và tiết kiệm 70% VRAM)

In [ ]:
%%capture
!pip install pip --upgrade
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

from unsloth import FastLanguageModel
import torch

# CHỌN KÍCH THƯỚC MÔ HÌNH STUDENT:
# - "0.5B" : Siêu nhẹ (~400MB GGUF), chạy mượt trên laptop 4GB RAM không GPU rời (Khuyên dùng)
# - "1.5B" : Cân bằng (~1.0GB GGUF), phù hợp máy 8GB+ RAM
# - "3B"   : Mạnh nhất (~2.0GB GGUF), phù hợp máy có GPU hoặc 16GB+ RAM
MODEL_SIZE = "0.5B"
MODEL_MAP = {
    "0.5B": "unsloth/Qwen2.5-Coder-0.5B-Instruct-bnb-4bit",
    "1.5B": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "3B":   "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
}
model_name = MODEL_MAP[MODEL_SIZE]

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print(f">>> Nạp mô hình nền {model_name} thành công!")

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Độ dài chuỗi tối đa hỗ trợ (đủ cho Spec + Verilog + TB)
dtype = None           # None: Tự động nhận diện (Float16 cho T4)
load_in_4bit = True    # 4-bit quantization giúp model 3B chỉ tốn ~5.5GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print(">>> Nạp mô hình Qwen2.5-Coder-3B thành công!")

## Bước 4: Cấu hình tham số LoRA (Low-Rank Adaptation)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank LoRA: 16 là mức tối ưu cân bằng độ chính xác và tốc độ
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth tối ưu tốt nhất khi dropout = 0
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Tiết kiệm tối đa VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print(">>> Đã gắn LoRA Adapters vào Qwen2.5-Coder-3B!")

## Bước 5: Cấu trúc Chat Template & Chuẩn bị dữ liệu mẫu (Dummy Dataset)
*Đây chính là format dữ liệu chuẩn mà Nhóm 1 sẽ xuất ra từ VerilogDB.*

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        user_content = f"{instruction}\n\n[Đặc tả thiết kế]:\n{input_text}" if input_text else instruction
        messages = [
            {"role": "system", "content": "Bạn là chuyên gia thiết kế vi mạch RTL và Verilog synthesizable chuẩn công nghiệp IEEE 1364-2001. Luôn sinh mã chuẩn sạch, đầy đủ cổng, không tạo chốt ẩn (no latches)."},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
        texts.append(text)
    return { "text" : texts }

# Dữ liệu mẫu kiểm thử vòng lặp (Fallback Data)
sample_data = [
    {
        "instruction": "Hãy thiết kế bộ đếm nhị phân 4-bit đồng bộ có reset tích cực mức thấp và chân cho phép đếm en.",
        "input": "Inputs: clk, rst_n, en. Outputs: [3:0] count.",
        "output": "```verilog\nmodule counter_4bit (\n    input  wire       clk,\n    input  wire       rst_n,\n    input  wire       en,\n    output reg  [3:0] count\n);\n    always @(posedge clk or negedge rst_n) begin\n        if (!rst_n) begin\n            count <= 4'd0;\n        end else if (en) begin\n            count <= count + 1'b1;\n        end\n    end\nendmodule\n```"
    }
]

import json, urllib.request, os
dataset_url = "https://raw.githubusercontent.com/LamGianggg/TKLLS/main/training/dataset_clean_verified.json"
dataset_file = "dataset_clean_verified.json"
training_data = sample_data

try:
    if not os.path.exists(dataset_file):
        print(">>> Đang tải tập dữ liệu sạch từ Nhóm 1 (GitHub)...")
        urllib.request.urlretrieve(dataset_url, dataset_file)
    if os.path.exists(dataset_file):
        with open(dataset_file, "r", encoding="utf-8") as f:
            training_data = json.load(f)
        print(f">>> Nạp thành công {len(training_data)} mẫu Verilog sạch 100% đạt chuẩn ModelSim từ Nhóm 1!")
except Exception as e:
    print(f">>> Sử dụng dữ liệu dự phòng do lỗi tải: {e}")

dataset = Dataset.from_list(training_data)
dataset = dataset.map(formatting_prompts_func, batched = True)
print(">>> Đã nạp và format xong dữ liệu huấn luyện! Số lượng mẫu:", len(dataset))

## Bước 6: Khởi tạo SFTTrainer và Chạy thử huấn luyện (Train Loop Verification)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 20, # Bỏ comment nếu chỉ muốn chạy test nhanh 1-2 phút
        num_train_epochs = 1, # Huấn luyện toàn bộ 1.098 mẫu sạch (~137 bước, mất ~12 phút trên T4 GPU)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print(">>> Bắt đầu chạy thử huấn luyện LoRA...")
trainer_stats = trainer.train()
print(">>> HUẤN LUYỆN THỬ NGHIỆM THÀNH CÔNG RỰC RỠ!")

## Bước 7: Kiểm thử sinh mã Verilog sau huấn luyện (Inference Test)

In [ ]:
FastLanguageModel.for_inference(model) # Chuyển sang chế độ suy luận nhanh

test_messages = [
    {"role": "system", "content": "Bạn là chuyên gia thiết kế vi mạch RTL và Verilog synthesizable chuẩn công nghiệp IEEE 1364-2001. Luôn sinh mã chuẩn sạch, đầy đủ cổng, không tạo chốt ẩn (no latches)."},
    {"role": "user", "content": "Hãy thiết kế bộ đếm 8-bit có reset đồng bộ tích cực cao và ngõ ra cờ tràn tc khi count = 255.\n\n[Đặc tả thiết kế]:\nInputs: clk, rst. Outputs: [7:0] count, tc."}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 512, use_cache = True)

# Lưu LoRA adapter SFT (dùng nối tiếp sang bước DPO)
sft_dir = "spec2rtl_sft_lora"
model.save_pretrained(sft_dir)
tokenizer.save_pretrained(sft_dir)
print(f">>> Đã lưu SFT LoRA adapter vào thư mục: {sft_dir}")

!zip -r spec2rtl_sft_lora.zip spec2rtl_sft_lora
print(">>> Đã tạo file nén: spec2rtl_sft_lora.zip (giữ nguyên phiên Colab để chạy tiếp notebook DPO hoặc tải về!)")

In [ ]:
# Lưu LoRA adapter nhẹ
model.save_pretrained("qwen2.5_3b_verilog_lora")
tokenizer.save_pretrained("qwen2.5_3b_verilog_lora")
print(">>> Đã lưu LoRA adapter vào thư mục: qwen2.5_3b_verilog_lora")

# Nén zip để tải về máy trường nếu cần
!zip -r qwen2.5_3b_verilog_lora.zip qwen2.5_3b_verilog_lora
print(">>> Đã tạo file nén: qwen2.5_3b_verilog_lora.zip sẵn sàng tải về!")